In [1]:
!pip install git+https://github.com/huggingface/transformers.git
!pip install accelerate

  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-q_yxtws4
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-q_yxtws4
  Resolved https://github.com/huggingface/transformers.git to commit 94df0e65602922be2831b3faa457a2bde78b936b
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.0/502.0 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 71.9 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.0.0.dev0-py3-none-any.whl size=11386959 sha256=8852975d5e5ac735415b047a95a8a3bd27691ab6f8538b6c24fa5bdd22049459
  Stored in directory: /tmp/pip-ephem-wheel-cache-51883iug/wheels/32/4b/78/f195c684dd3a9ed21f3b39fe8f85b48df7918581b6437be143
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found

In [2]:
# =============================================================================
# 1. IMPORTS & SETUP
# =============================================================================
import os
import re
import pandas as pd
import polars as pl
import torch
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score

# Use the pipeline function for a higher-level interface
from transformers import pipeline

# Access Hugging Face token from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
except:
    print("HF_TOKEN secret not found.")
    os.environ["HF_TOKEN"] = "YOUR_HF_TOKEN_HERE"


# =============================================================================
# 2. CONFIGURATION
# =============================================================================
class Config:
    # --- Model Configuration ---
    # Using the model name from the official code snippet
    MODEL_NAME = "Equall/Saul-Instruct-v1"

    # --- Data Paths ---
    POS_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair.txt"
    NEG_DATA_FILE = "/kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt"
    OUTPUT_FILE = "/kaggle/working/zero_shot_results_instruct.csv"

    # --- Inference Configuration ---
    SAMPLE_SIZE = 500
    BATCH_SIZE = 4 # You can increase this with pipelines, e.g., to 8

    # --- Environment ---
    DEVICE = 0 # Pipelines manage device mapping, we can specify the primary GPU


# =============================================================================
# 3. DATA LOADING & PREPARATION (Unchanged from last version)
# =============================================================================
FILENAME_PATTERN = re.compile(r"SupremeCourt_(\d{4})_(\d+)\.txt")

def parse_lrec_line(line: str):
    parts = line.strip().split("\t")
    fname_indices = [i for i, p in enumerate(parts) if p.endswith(".txt")]
    if len(fname_indices) != 2: return None
    try:
        sentpair_id = int(parts[0])
        sent1 = " ".join(parts[fname_indices[0] + 2 : fname_indices[1]]).strip('"')
        sent2 = " ".join(parts[fname_indices[1] + 2 : len(parts) - 2]).strip('"')
        label = parts[-1]
    except (ValueError, IndexError): return None
    return {"id": sentpair_id, "sent1": sent1, "sent2": sent2, "label": label}

def load_lrec_file(filepath: str) -> pd.DataFrame:
    rows = []
    print(f"Reading data from {filepath}...")
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if parsed_row := parse_lrec_line(line):
                rows.append(parsed_row)
    return pd.DataFrame(rows)

def load_and_split_data(config: Config):
    print("Loading and splitting data...")
    df_pos = load_lrec_file(config.POS_DATA_FILE)
    df_neg = load_lrec_file(config.NEG_DATA_FILE)
    if df_pos.empty or df_neg.empty:
        print("ERROR: Data files failed to load.")
        return None
    df_pos_sample = df_pos.sample(n=min(10000, len(df_pos)), random_state=42)
    df_neg_sample = df_neg.sample(n=min(10000, len(df_neg)), random_state=42)
    remaining_pos = df_pos.drop(df_pos_sample.index)
    remaining_neg = df_neg.drop(df_neg_sample.index)
    df_test = pd.concat([remaining_pos, remaining_neg]).reset_index(drop=True)
    df_test = df_test.dropna(subset=['id', 'sent1', 'sent2', 'label'])
    print(f"Created test set with {len(df_test)} samples.")
    if config.SAMPLE_SIZE > 0:
        print(f"Using a sample of {config.SAMPLE_SIZE} examples from the test set for evaluation.")
        df_test = df_test.sample(n=min(config.SAMPLE_SIZE, len(df_test)), random_state=42).reset_index(drop=True)
    return df_test

# =============================================================================
# 4. ZERO-SHOT INFERENCE PIPELINE (REWRITTEN)
# =============================================================================
class ZeroShotClassifierWithPipeline:


    PROMPT_TEMPLATE = """You are an expert legal analyst specializing in argumentation theory, specifically Walton's model of argument schemes. Your task is to analyze the relationship between two given sentences from a legal context (Indian Supreme Court Judgemment) and classify this relationship as SUPPORT, ATTACK, or NO RELATION.

First, provide a brief, step-by-step reasoning that explains your analysis. Then, on a new line, state the final classification.

Here are the definitions of the categories:
- SUPPORT: Sentence 2 provides evidence, a premise, a justification, or a logical conclusion that strengthens or affirms the claim made in Sentence 1.
- ATTACK: Sentence 2 provides a counter-argument, rebuttal, exception, or evidence that weakens, limits, or contradicts the claim made in Sentence 1.
- NO RELATION: The two sentences are on different topics or present unrelated facts, and one does not logically impact the other within an argumentative structure.

---
**Example 1:**
Sentence 1: "The Tribunal has therefore misdirected itself in law in arriving at its finding , and in refusing to require the Tribunal to state the case and to refer it , the High Court was , in our view , in error ."
Sentence 2: "The appeal is therefore allowed and the proceedings are remanded to the High Court with a direction to proceed according to law ."
Reasoning: Sentence 1 identifies a legal error made by the High Court and the Tribunal. Sentence 2 states the direct legal consequence and remedy for that error: allowing the appeal and remanding the case. Sentence 2 is the logical outcome of the premise in Sentence 1.
Classification: SUPPORT
---
**Example 2:**
Sentence 1: "The conviction for these two offences is based on the assumption that the petitioner was a member of the unlawful assembly but his acquittal in respect of the charge under Section 148 , I. P. C. , must necessarily lead to the inference that he was not a member of such an assembly ."
Sentence 2: "Hence , his conviction under Sections 455 and 323 , I. P. C. , read with Section 149 , I. P. C. , must be set aside ."
Reasoning: Sentence 1 establishes a critical premise: an acquittal on one charge proves the petitioner was not part of an unlawful assembly. Sentence 2 draws the necessary legal conclusion from this premise, stating that other convictions based on membership of that assembly must be overturned.
Classification: SUPPORT
---
**Example 3:**
Sentence 1: "It is , however , observed that most of the cases of permanent absorption are those where the officers were taken on deputation initially under the method of ` transfer on deputation/transfer ' contained in the relevant recruitment rules ."
Sentence 2: "The undersigned is directed to say that the existing instructions on seniority of transferees contained in paras 7 of the Annexure to this Department 's O.M. No. 9/11/55-RPs dated the 22nd December , 1959 ( copy enclosed ) mainly deal with cases where persons are straight way appointed on transfer ."
Reasoning: Sentence 1 describes a common practice for permanent absorption. Sentence 2 points out that the existing official instructions are for a different scenario (straight transfers). This highlights a mismatch or conflict, challenging the applicability of the existing rules to the practice described in Sentence 1.
Classification: ATTACK
---
**Example 4:**
Sentence 1: "The High Court observed that it was open to the Corporation to take appropriate proceedings for recovery of the dues claimed by the Corporation ."
Sentence 2: "If it is so , the appellant- Corporation , in our opinion , is right in submitting that the proceedings could have been continued after the retirement of the respondent-employee as far as the financial loss caused to the Corporation because of negligence on the part of employee and the benefit claimed by the respondent-workman on his terminal benefits ."
Reasoning: Sentence 1 presents a general permission noted by the High Court. Sentence 2 argues for a very specific and aggressive interpretation of that permission—that proceedings can continue even after retirement. This specific assertion challenges any potentially narrower or more limited interpretation of the "appropriate proceedings" mentioned in Sentence 1.
Classification: ATTACK
---
**Example 5:**
Sentence 1: "The defendant was found guilty of perjury under Section 191 of the Indian Penal Code."
Sentence 2: "The new amendment to the Companies Act 2013 will come into effect next month."
Reasoning: Sentence 1 states a fact about a criminal conviction under the Indian Penal Code. Sentence 2 states a fact about a future change in corporate law. These two statements are from entirely different legal domains and have no logical or argumentative connection to each other.
Classification: NO RELATION
---
**Example 6:**
Sentence 1: "We can not , therefore , say that the petitioner was arbitrarily or unfairly treated or that equality was denied to him when he was transferred from the post of Chief Secretary and in his place Sabanayagam , his junior , was promoted and confirmed ."
Sentence 2: "The Government of India by a letter dated 26 September , 1969 stated that the status of Chief Secretary as the head of the Secretariat Organisation in the State should remain unquestioned ."
Reasoning: Sentence 1 makes a specific judgment about the fairness of a particular petitioner's transfer from the Chief Secretary post. Sentence 2 makes a general declaration about the status of that post. The general statement in Sentence 2 does not provide evidence to either support or attack the specific legal conclusion in Sentence 1.
Classification: NO RELATION
---
Now, analyze the following sentences:

Sentence 1: "{sent1}"
Sentence 2: "{sent2}"
"""
                   
    def __init__(self, config: Config):
        self.config = config
        self.pipe = self._load_pipeline()
        if self.pipe.tokenizer.pad_token_id is None:
            self.pipe.tokenizer.pad_token_id = self.pipe.model.config.eos_token_id

    def _load_pipeline(self):
        print(f"Loading pipeline for model {self.config.MODEL_NAME}...")
        # This single command initializes the model, tokenizer, and puts it on the GPU
        return pipeline(
            "text-generation",
            model=self.config.MODEL_NAME,
            dtype=torch.bfloat16,
            device_map="auto",
            trust_remote_code=True # Good practice to keep this
        )

    def create_message(self, sent1: str, sent2: str) -> list:
        """Creates the message structure for the chat template."""
        content = self.PROMPT_TEMPLATE.format(sent1=sent1, sent2=sent2)
        return [{"role": "user", "content": content}]

    def parse_output(self, generated_text: str) -> str:
        """Extracts the label from the model's free-text response."""
        output = generated_text.strip().upper()
        if "SUPPORT" in output: return "SUPPORT"
        if "ATTACK" in output: return "ATTACK"
        if "NO RELATION" in output or "NO_REL" in output: return "NO_REL"
        return "UNKNOWN"

    def predict(self, df: pd.DataFrame):
        """Runs batch inference on the entire dataframe using the pipeline."""
        all_messages = [self.create_message(row.sent1, row.sent2) for _, row in df.iterrows()]
        
        # Format all messages using the model's specific chat template
        prompts = [
            self.pipe.tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
            for msg in all_messages
        ]
        
        all_predictions = []
        print(f"Starting inference on {len(prompts)} samples...")
        
        # The pipeline can process a list of prompts in a batch
        outputs = self.pipe(
            prompts,
            max_new_tokens=256,
            do_sample=False,
            batch_size=self.config.BATCH_SIZE,
            return_full_text=False # Only return the generated part
        )
        
        # Extract and parse the generated text from each output
        for output in outputs:
            generated_text = output[0]['generated_text']
            prediction = self.parse_output(generated_text)
            all_predictions.append(prediction)
            
        return all_predictions

# =============================================================================
# 5. MAIN EXECUTION BLOCK
# =============================================================================
if __name__ == '__main__':
    config = Config()
    df_test = load_and_split_data(config)

    if df_test is not None:
        classifier = ZeroShotClassifierWithPipeline(config)
        predictions = classifier.predict(df_test)

        # --- EVALUATION ---
        print("\n" + "="*50)
        print("      ZERO-SHOT CLASSIFICATION ON TEST SET")
        print("="*50)
        true_labels = df_test['label']
        accuracy = accuracy_score(true_labels, predictions)
        print(f"\nOverall Accuracy: {accuracy:.4f}\n")
        print("Classification Report (Precision, Recall, F1-Score):")
        report = classification_report(
            true_labels, predictions, labels=list(true_labels.unique()), zero_division=0
        )
        print(report)

        # --- SAVING THE OUTPUT ---
        output_df = pd.DataFrame({
            'pair_id': df_test['id'],
            'sentence_1': df_test['sent1'],
            'sentence_2': df_test['sent2'],
            'original_label': df_test['label'],
            'llm_generated_label': predictions
        })
        output_df.to_csv(config.OUTPUT_FILE, index=False)
        print(f"\n Results saved successfully to {config.OUTPUT_FILE}")

        # Display some examples
        print("\n--- Sample Predictions ---")
        print(output_df.head(10))

Loading and splitting data...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair.txt...
Reading data from /kaggle/input/lrec-tcs-attack-support/sentencePair_neg.txt...
Created test set with 20506 samples.
Using a sample of 500 examples from the test set for evaluation.
Loading pipeline for model Equall/Saul-Instruct-v1...


config.json:   0%|          | 0.00/653 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

model-00004-of-00006.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Starting inference on 500 samples...


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for o


      ZERO-SHOT CLASSIFICATION ON TEST SET

Overall Accuracy: 0.3000

Classification Report (Precision, Recall, F1-Score):
              precision    recall  f1-score   support

      NO_REL       0.79      0.14      0.23       247
      ATTACK       0.43      0.18      0.25       130
     SUPPORT       0.25      0.76      0.37       123

   micro avg       0.32      0.30      0.31       500
   macro avg       0.49      0.36      0.29       500
weighted avg       0.56      0.30      0.27       500


 Results saved successfully to /kaggle/working/zero_shot_results_instruct.csv

--- Sample Predictions ---
   pair_id                                         sentence_1  \
0    15136  It was , however , held that in case the Junio...   
1     4842  However , the charges Nos. II , V and X were e...   
2    10249  For the price of goods is affected by many fac...   
3     9450  Mr. Andley also attempted to argue that the de...   
4    15504  Learned counsel for the petitioners emphatical...  